In [ ]:
import pandas as pd
import os
import re
from pathlib import Path
import numpy as np

INPUT_DIR  = r"C:\Users\rashe\Desktop\Cattle Scan\Heat Alert\Farm 85 MOVE"
OUTPUT_DIR = r"C:\Users\rashe\Desktop\Cattle Scan\Heat Alert\Fixed Farm 85"

TEMP_THRESHOLD  = 30.0  
MIN_CONSECUTIVE = 3     


def get_output_filename(original_name: str) -> str:
    match = re.search(r'animalId-(\d+)-farmId-(\d+)', original_name)
    if match:
        return f"animal-id-{match.group(1)}-farm-id-{match.group(2)}.csv"
    return original_name


def find_coldstart_cutoff(df: pd.DataFrame):
  
    temps = df[['date', 'temp']].copy()
    temps['temp'] = pd.to_numeric(temps['temp'], errors='coerce')

   
    valid = temps.dropna(subset=['temp']).reset_index(drop=True)

    if len(valid) == 0:
        return None

    first_n = valid.head(MIN_CONSECUTIVE)
    if not (first_n['temp'] < TEMP_THRESHOLD).all():
        return None


    df2 = df.copy()
    df2['_date'] = pd.to_datetime(df2['date']).dt.date
    df2['temp']  = pd.to_numeric(df2['temp'], errors='coerce')

    for d in sorted(df2['_date'].unique()):
        day_temps = df2[df2['_date'] == d]['temp'].dropna()
        if len(day_temps) == 0:
            continue
        if (day_temps >= TEMP_THRESHOLD).all():
            return pd.Timestamp(d)

    return None


def process_file(csv_path: str, output_path: str):
    df = pd.read_csv(csv_path, sep=',')

    col_map = {c.lower(): c for c in df.columns}
    date_col = col_map.get('date', None)
    temp_col = col_map.get('temp', None)

    if date_col is None or temp_col is None:
        print(f"  [SKIP] Missing 'date' or 'temp' column in {os.path.basename(csv_path)}")
        return 0

    df = df.rename(columns={date_col: 'date', temp_col: 'temp'})
    original_len = len(df)

    cutoff = find_coldstart_cutoff(df)

    if cutoff is not None:
        df['_date'] = pd.to_datetime(df['date']).dt.date
        df = df[df['_date'] >= cutoff.date()]
        df = df.drop(columns=['_date'])
        removed = original_len - len(df)
        print(f"  Removed {removed} rows (cold-start sensor error). Clean data starts: {cutoff.date()}")
    else:
        print(f"  No cold-start error detected. File untouched.")

    df.to_csv(output_path, index=False)
    return original_len - len(df)


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    csv_files = list(Path(INPUT_DIR).glob("*.csv"))
    if not csv_files:
        print(f"No CSV files found in:\n  {INPUT_DIR}")
        return

    print(f"Found {len(csv_files)} file(s). Processing...\n")
    total_removed = 0

    for csv_path in sorted(csv_files):
        out_name = get_output_filename(csv_path.name)
        out_path = os.path.join(OUTPUT_DIR, out_name)
        print(f"[{csv_path.name}]")
        removed = process_file(str(csv_path), out_path)
        total_removed += removed
        print(f"  → Saved: {out_name}\n")

    print(f"Done. Total rows removed across all files: {total_removed}")
    print(f"Clean files saved to:\n  {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

Found 104 file(s). Processing...

[animalId-27187-farmId-85 2026-02-20T22_00 2026-03-24T06_07.csv]
  Removed 200 rows (cold-start sensor error). Clean data starts: 2026-02-23
  → Saved: animal-id-27187-farm-id-85.csv

[animalId-2820-farmId-85 2026-02-18T22_00 2026-03-24T05_05.csv]
  Removed 104 rows (cold-start sensor error). Clean data starts: 2026-02-20
  → Saved: animal-id-2820-farm-id-85.csv

[animalId-3236-farmId-85 2026-02-18T22_00 2026-03-24T05_06.csv]
  Removed 104 rows (cold-start sensor error). Clean data starts: 2026-02-20
  → Saved: animal-id-3236-farm-id-85.csv

[animalId-3241-farmId-85 2026-02-20T22_00 2026-03-24T05_06.csv]
  Removed 104 rows (cold-start sensor error). Clean data starts: 2026-02-22
  → Saved: animal-id-3241-farm-id-85.csv

[animalId-3253-farmId-85 2026-02-20T22_00 2026-03-24T05_07.csv]
  Removed 104 rows (cold-start sensor error). Clean data starts: 2026-02-22
  → Saved: animal-id-3253-farm-id-85.csv

[animalId-3314-farmId-85 2026-02-18T22_00 2026-03-24T0

C:\Users\rashe\AppData\Local\Temp\ipykernel_23816\3076082830.py:60: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path, sep=',')


  → Saved: animal-id-7686-farm-id-85.csv

[animalId-7699-farmId-85 2026-02-16T22_00 2026-03-24T06_00.csv]
  Removed 104 rows (cold-start sensor error). Clean data starts: 2026-02-18
  → Saved: animal-id-7699-farm-id-85.csv

[animalId-80-farmId-85 2026-02-18T22_00 2026-03-24T05_02.csv]
  Removed 104 rows (cold-start sensor error). Clean data starts: 2026-02-20
  → Saved: animal-id-80-farm-id-85.csv

[animalId-8003-farmId-85 2025-04-01T21_00 2026-03-24T06_01.csv]
  Removed 108 rows (cold-start sensor error). Clean data starts: 2025-04-03
  → Saved: animal-id-8003-farm-id-85.csv

[animalId-8005-farmId-85 2025-04-01T21_00 2026-03-24T06_02.csv]
  Removed 108 rows (cold-start sensor error). Clean data starts: 2025-04-03
  → Saved: animal-id-8005-farm-id-85.csv

[animalId-8038-farmId-85 2025-06-05T21_00 2026-03-24T06_03.csv]
  Removed 588 rows (cold-start sensor error). Clean data starts: 2025-06-12
  → Saved: animal-id-8038-farm-id-85.csv

[animalId-8046-farmId-85 2025-06-09T21_00 2026-03-24

In [ ]:
import re
import os
from pathlib import Path
import glob

def preprocess_file(file_path, output_dir):

    try:
        # Read the CSV file
        print(f"Processing: {os.path.basename(file_path)}")
        df = pd.read_csv(file_path)
        
        # Extract animal ID from filename
        file = os.path.basename(file_path)
        match = re.search(r'animal-id-(\d+)', file)
        if not match:
            print(f"  ⚠ Skipping {file} - Could not extract animal ID")
            return False
        
        animal_id = int(match.group(1))
        
        # Extract farm ID from filename
        farm_match = re.search(r'farm-id-(\d+)', file)
        farm_id = int(farm_match.group(1)) if farm_match else 5
        
        # Extract optional suffix (e.g. -2, -H) to avoid overwriting files with same animal+farm ID
        suffix_match = re.search(r'farm=id-\d+(-[^.]+)?\.csv', file)
        suffix = suffix_match.group(1) if suffix_match and suffix_match.group(1) else ""

        # Step 1: Drop columns
        columns_to_drop = ['waterTemp','aveHerdTemp','waterIntake']
        df = df.drop(columns=[col for col in columns_to_drop if col in df.columns], errors='ignore')
        
        # Step 2: Rename columns
        df = df.rename(columns={'date': 'datetime'})
        
        # Step 3: Interpolate missing values
        if 'humidity' in df.columns:
            df['humidity'] = df['humidity'].interpolate(method='linear')
        if 'airTemp' in df.columns:
            df['airTemp'] = df['airTemp'].interpolate(method='linear')
        if 'accel' in df.columns:
            df['accel'] = df['accel'].interpolate(method='linear')
        
        # Step 4: Fill leading NULLs with backward fill
        if 'airTemp' in df.columns:
            df['airTemp'] = df['airTemp'].bfill()
        if 'humidity' in df.columns:
            df['humidity'] = df['humidity'].bfill()
        if 'accel' in df.columns:
            df['accel'] = df['accel'].bfill()
       
        # Step 5: Process datetime and extract date/time
        df["datetime"] = pd.to_datetime(df["datetime"], format='mixed', errors='coerce')
        df["date"] = df["datetime"].dt.strftime('%Y-%m-%d')
        df["time"] = df["datetime"].dt.strftime('%H:%M:%S')
        
        # Step 6: Add animal ID column
        df['id'] = animal_id
        
        # Step 7: Process rumination
        if 'rumination' in df.columns:
            df['rumination'] = df['rumination'].interpolate(method='nearest')
            df['rumination'] = df['rumination'].fillna(0)
            df['rumination'] = df['rumination'].round().astype(int)
        
        # Step 8: Fix faulty temperature readings
        if 'temp' in df.columns:
            def fill_faulty_temps_smooth(df, faulty_threshold=35):
                df_copy = df.copy()
                df_copy.loc[df_copy['temp'] < faulty_threshold, 'temp'] = np.nan
                df_copy['temp'] = df_copy['temp'].interpolate(method='linear')
                df_copy['temp'] = df_copy['temp'].bfill().ffill()
                return df_copy
            
            df = fill_faulty_temps_smooth(df, faulty_threshold=35)
        
        # Step 9: Calculate THI (Temperature-Humidity Index)
        if 'airTemp' in df.columns and 'humidity' in df.columns:
            df['airTemp'] = pd.to_numeric(df['airTemp'], errors='coerce')
            df['humidity'] = pd.to_numeric(df['humidity'], errors='coerce')
            
            df['THI'] = (
                0.8 * df['airTemp']
                + 0.01 * df['humidity'] * (df['airTemp'] - 14.4)
                + 46.4
            ).round(2)
        
        # Step 10: Calculate daily rumination
        if 'rumination' in df.columns:
            daily_rumination = df.groupby(['date', 'id']).agg({
                'rumination': 'sum',
                'ageInDays': 'mean'
            }).reset_index()
            
            daily_rumination['rumination_minutes'] = daily_rumination['rumination'] * 15
            
            df = df.merge(
                daily_rumination[['date', 'id', 'rumination_minutes']],
                on=['date', 'id'],
                how='left'
            )
            
            df.rename(columns={'rumination_minutes': 'rumination_per_day'}, inplace=True)
        
        # Step 11: Calculate hourly average temperature
        if 'temp' in df.columns:
            df['datetime'] = pd.to_datetime(df['datetime'])
            df = df.sort_values('datetime').reset_index(drop=True)
            
            df['aveHerdTemp_hourly_avg'] = df['temp'].rolling(window=4, min_periods=4).mean()
            df['aveHerdTemp_hourly_avg'] = df['aveHerdTemp_hourly_avg'].bfill()
            df['aveHerdTemp'] = df['aveHerdTemp_hourly_avg']
            df.drop(columns=['aveHerdTemp_hourly_avg'], inplace=True)
        
        #Step 12: Reorder columns
        desired_order = ['datetime', 'event', 'temp', 'airTemp', 'humidity', 'accel',
                        'rumination', 'ageInDays', 'waterTemp', 'lactationNumber', 
                         'date', 'time', 'id', 'THI', 'rumination_per_day','aveHerdTemp','aveGroupTemp','aveHerdActivity',
                         'aveGroupActivity','aveHerdRumination','aveGroupRumination']
        # desired_order = ['datetime', 'event', 'temp', 'airTemp', 'humidity', 'accel',
        #                 'rumination', 'ageInDays', 'waterTemp', 'lactationNumber', 
        #                  'date', 'time', 'id', 'THI', 'rumination_per_day','aveHerdTemp']
        
        # Only include columns that exist in the dataframe
        existing_columns = [col for col in desired_order if col in df.columns]
        df = df[existing_columns]
        
        # Step 13: Save processed file (suffix preserves -2, -H variants)
        output_filename = f"processed-animalId-{animal_id}-farmId-{farm_id}{suffix}.csv"
        output_path = os.path.join(output_dir, output_filename)
        df.to_csv(output_path, index=False)
        
        print(f"  ✓ Saved: {output_filename}")
        return True
        
    except Exception as e:
        print(f"  ✗ Error processing {os.path.basename(file_path)}: {str(e)}")
        return False


def batch_preprocess(directory_path, output_path):
    """
    Batch process all CSV files in the directory.
    
    Parameters:
    - directory_path: Path to directory containing CSV files
    - output_path: Path to directory where processed files will be saved
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_path, exist_ok=True)
    print(f"Output directory: {output_path}")
    
    # Get all CSV files in the directory
    csv_files = glob.glob(os.path.join(directory_path, "*.csv"))
    
    if not csv_files:
        print(f"No CSV files found in {directory_path}")
        return
    
    print(f"\nFound {len(csv_files)} CSV files to process\n")
    print("=" * 60)
    
    # Process each file
    successful = 0
    failed = 0
    
    for file_path in csv_files:
        if preprocess_file(file_path, output_path):
            successful += 1
        else:
            failed += 1
        print()
    
    # Summary
    print("=" * 60)
    print(f"\nProcessing Complete!")
    print(f"  ✓ Successful: {successful}")
    print(f"  ✗ Failed: {failed}")
    print(f"  Total: {len(csv_files)}")
    print(f"\nProcessed files saved to: {output_path}")


if __name__ == "__main__":
    # Set your paths here
    directory_path = r"C:\Users\rashe\Desktop\Cattle Scan\Heat Alert\Fixed Farm 85"
    output_path    = r"C:\Users\rashe\Desktop\Cattle Scan\Heat Alert\Processed Data 85"

    batch_preprocess(directory_path, output_path)

Output directory: C:\Users\rashe\Desktop\Cattle Scan\Heat Alert\Processed Data 85

Found 104 CSV files to process

Processing: animal-id-27187-farm-id-85.csv
  ✓ Saved: processed-animalId-27187-farmId-85.csv

Processing: animal-id-2820-farm-id-85.csv
  ✓ Saved: processed-animalId-2820-farmId-85.csv

Processing: animal-id-3236-farm-id-85.csv
  ✓ Saved: processed-animalId-3236-farmId-85.csv

Processing: animal-id-3241-farm-id-85.csv
  ✓ Saved: processed-animalId-3241-farmId-85.csv

Processing: animal-id-3253-farm-id-85.csv
  ✓ Saved: processed-animalId-3253-farmId-85.csv

Processing: animal-id-3314-farm-id-85.csv
  ✓ Saved: processed-animalId-3314-farmId-85.csv

Processing: animal-id-3359-farm-id-85.csv
  ✓ Saved: processed-animalId-3359-farmId-85.csv

Processing: animal-id-3414-farm-id-85.csv
  ✓ Saved: processed-animalId-3414-farmId-85.csv

Processing: animal-id-3476-farm-id-85.csv
  ✓ Saved: processed-animalId-3476-farmId-85.csv

Processing: animal-id-3477-farm-id-85.csv
  ✓ Saved: pr

C:\Users\rashe\AppData\Local\Temp\ipykernel_23816\2207367525.py:20: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


  ✓ Saved: processed-animalId-7686-farmId-85.csv

Processing: animal-id-7699-farm-id-85.csv
  ✓ Saved: processed-animalId-7699-farmId-85.csv

Processing: animal-id-80-farm-id-85.csv
  ✓ Saved: processed-animalId-80-farmId-85.csv

Processing: animal-id-8003-farm-id-85.csv
  ✓ Saved: processed-animalId-8003-farmId-85.csv

Processing: animal-id-8005-farm-id-85.csv
  ✓ Saved: processed-animalId-8005-farmId-85.csv

Processing: animal-id-8038-farm-id-85.csv
  ✓ Saved: processed-animalId-8038-farmId-85.csv

Processing: animal-id-8046-farm-id-85.csv
  ✓ Saved: processed-animalId-8046-farmId-85.csv

Processing: animal-id-9044-farm-id-85.csv
  ✓ Saved: processed-animalId-9044-farmId-85.csv

Processing: animal-id-9052-farm-id-85.csv
  ✓ Saved: processed-animalId-9052-farmId-85.csv

Processing: animal-id-9112-farm-id-85.csv
  ✓ Saved: processed-animalId-9112-farmId-85.csv

Processing: animal-id-9131-farm-id-85.csv
  ✓ Saved: processed-animalId-9131-farmId-85.csv


Processing Complete!
  ✓ Successf

In [5]:
import os
import glob
import pandas as pd

def merge_processed_files(input_dir, farm_id=85):
    pattern = os.path.join(input_dir, "processed-*.csv")
    files = glob.glob(pattern)

    if not files:
        print(f"No files found matching: {pattern}")
        return

    dfs = []
    for fp in files:
        try:
            df = pd.read_csv(fp)

            if "datetime" in df.columns:
                df["datetime"] = pd.to_datetime(df["datetime"], format="mixed", errors="coerce")

            dfs.append(df)
            print(f"Loaded: {os.path.basename(fp)}  rows={len(df)}")

        except Exception as e:
            print(f"Skipping {os.path.basename(fp)} due to error: {e}")

    if not dfs:
        print("No files loaded successfully.")
        return

    merged = pd.concat(dfs, ignore_index=True)

    # Ensure columns exist before sorting
    sort_cols = []
    if "id" in merged.columns:
        sort_cols.append("id")
    if "datetime" in merged.columns:
        sort_cols.append("datetime")

    if sort_cols:
        merged = merged.sort_values(sort_cols).reset_index(drop=True)

    out_path = os.path.join(input_dir, f"merged-data-farm-id-{farm_id}.csv")
    merged.to_csv(out_path, index=False)

    print("\n Done")
    print(f"Total files merged: {len(dfs)}")
    print(f"Total rows: {len(merged)}")
    print(f"Saved to: {out_path}")

if __name__ == "__main__":
    input_dir = r"C:\Users\rashe\Desktop\Cattle Scan\Heat Alert\Processed Data 85"
    merge_processed_files(input_dir, farm_id=85)

Loaded: processed-animalId-27187-farmId-85.csv  rows=2802
Loaded: processed-animalId-2820-farmId-85.csv  rows=3084
Loaded: processed-animalId-3236-farmId-85.csv  rows=3062
Loaded: processed-animalId-3241-farmId-85.csv  rows=2882
Loaded: processed-animalId-3253-farmId-85.csv  rows=2882
Loaded: processed-animalId-3314-farmId-85.csv  rows=3084
Loaded: processed-animalId-3359-farmId-85.csv  rows=28830
Loaded: processed-animalId-3414-farmId-85.csv  rows=2708
Loaded: processed-animalId-3476-farmId-85.csv  rows=3066
Loaded: processed-animalId-3477-farmId-85.csv  rows=2916
Loaded: processed-animalId-3478-farmId-85.csv  rows=2880
Loaded: processed-animalId-3490-farmId-85.csv  rows=2948
Loaded: processed-animalId-3501-farmId-85.csv  rows=2900
Loaded: processed-animalId-3508-farmId-85.csv  rows=2882
Loaded: processed-animalId-3529-farmId-85.csv  rows=3094
Loaded: processed-animalId-3540-farmId-85.csv  rows=3048
Loaded: processed-animalId-3541-farmId-85.csv  rows=2874
Loaded: processed-animalId-35

C:\Users\rashe\AppData\Local\Temp\ipykernel_23816\2471527178.py:16: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(fp)


Loaded: processed-animalId-7686-farmId-85.csv  rows=32824
Loaded: processed-animalId-7699-farmId-85.csv  rows=3267
Loaded: processed-animalId-80-farmId-85.csv  rows=3020
Loaded: processed-animalId-8003-farmId-85.csv  rows=30552
Loaded: processed-animalId-8005-farmId-85.csv  rows=27837
Loaded: processed-animalId-8038-farmId-85.csv  rows=26584
Loaded: processed-animalId-8046-farmId-85.csv  rows=26444
Loaded: processed-animalId-9044-farmId-85.csv  rows=2904
Loaded: processed-animalId-9052-farmId-85.csv  rows=2704
Loaded: processed-animalId-9112-farmId-85.csv  rows=2998
Loaded: processed-animalId-9131-farmId-85.csv  rows=2790

 Done
Total files merged: 104
Total rows: 1179878
Saved to: C:\Users\rashe\Desktop\Cattle Scan\Heat Alert\Processed Data 85\merged-data-farm-id-85.csv


In [7]:
df['event'].unique()

array([nan, 'MOVE', 'VAC', 'HOOF', 'OPEN', 'PREG', 'HOOF VAC',
       'HOOF PREG', 'DRY', 'FRESH MOVE', 'BRED', 'MOVE FRESH',
       'DRY MFRM MOVE', 'MFRM MOVE DRY', 'PREG VAC', 'FRESH', 'MOVE VAC'],
      dtype=object)

In [6]:
import pandas as pd

df = pd.read_csv("Processed Data 85/merged-data-farm-id-85.csv")
df.head()

C:\Users\rashe\AppData\Local\Temp\ipykernel_23816\796883036.py:3: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Processed Data 85/merged-data-farm-id-85.csv")


,datetime,event,temp,airTemp,humidity,accel,rumination,ageInDays,lactationNumber,date,time,id,THI,rumination_per_day,aveHerdTemp,aveGroupTemp,aveHerdActivity,aveGroupActivity,aveHerdRumination,aveGroupRumination
0,2026-02-18 00:11:39,NaN,39.2,9.1,81.6,0.09,1,1509,3,2026-02-18,00:11:39,47,49.36,450,39.500,38.73,0.13,0.12,0.17,0.33
1,2026-02-18 00:26:39,NaN,39.4,8.4,82.6,0.09,0,1509,3,2026-02-18,00:26:39,47,48.16,450,39.500,NaN,NaN,NaN,NaN,NaN
2,2026-02-18 00:41:39,NaN,39.7,8.2,82.4,0.07,0,1509,3,2026-02-18,00:41:39,47,47.85,450,39.500,NaN,NaN,NaN,NaN,NaN
3,2026-02-18 00:56:39,NaN,39.7,9.7,82.3,0.11,0,1509,3,2026-02-18,00:56:39,47,50.29,450,39.500,NaN,NaN,NaN,NaN,NaN
4,2026-02-18 01:11:39,NaN,39.7,9.8,80.6,0.09,0,1509,3,2026-02-18,01:11:39,47,50.53,450,39.625,39.37,0.12,0.10,0.23,0.50


In [8]:
df['farmId'] = 85
new_df = df.copy()
new_df.to_csv(r"C:\Users\rashe\Desktop\Cattle Scan\Heat Alert\Processed Data 85\merged-data-farm-id-85.csv", index=False)


In [9]:
#new_df count number of RUMINATION WATCHLIST and WATCHLIST RUMINATION in the dataset
df_count = df[df['event'].isin(['RUMINATION WATCHLIST', 'WATCHLIST RUMINATION'])]
new_df = df_count.groupby(['id', 'event']).size().reset_index(name='count')
len(new_df)


5

In [8]:


import pandas as pd
import os

file_path = r"C:\Users\rashe\Desktop\Cattle Scan\Heat Alert\Processed Data\merged-data-farm-id-76.csv"

# Load
df = pd.read_csv(file_path)

# Clean spacing first (fix 'HEAT ')
df["event"] = df["event"].astype(str).str.strip()

# Rename mapping
event_rename_map = {
    "Move": "MOVE",
    "HOOF HEAT": "HOOF",
    "HEAT HOOF": "HOOF",
    "Hoof Maintenance": "HOOF",
    "HEAT OPEN": "HEAT",
    "HEAT BRED": "HEAT",
    "BRED HEAT": "HEAT"
}

df["event"] = df["event"].replace(event_rename_map)

# Save updated file
output_path = r"C:\Users\rashe\Desktop\Cattle Scan\Heat Alert\Processed Data\updated-merged-data-farm-id-76.csv"
df.to_csv(output_path, index=False)

print("Events renamed and file saved successfully.")

Events renamed and file saved successfully.


In [9]:
df = pd.read_csv(r"C:\Users\rashe\Desktop\Cattle Scan\Heat Alert\Processed Data\updated-merged-data-farm-id-76.csv")

In [10]:
df['event'].unique()

array([nan, 'MOVE', 'Bred', 'OPEN', 'HEAT', 'PREG', 'PREG VAC', 'HOOF',
       'VAC', 'ABORT', 'BRED', 'Treatment', 'Vaccination',
       'PREG Preg Check', 'DRY', 'FRESH', 'ABORT OPEN'], dtype=object)

In [ ]:
datetime	event	temp	airTemp	humidity	accel	rumination	ageInDays	lactationNumber	date	time	id	THI	rumination_per_day	aveHerdTemp	aveGroupTemp	aveHerdActivity	aveGroupActivity	aveHerdRumination	aveGroupRumination
